In [1]:
import requests
import time
import json

# ========== 1. 新增：微信公众号配置 ==========
WECHAT_CONFIG = {
    "APPID": "wx0aa5d92b18cc1e02",          # 替换成你的测试号APPID
    "APPSECRET": "15901b2fdfbf610bc3903135b148900f",  # 替换成你的测试号APPSECRET
    "OPENID": "你的测试用户OpenID"       # 替换成你的测试用户OpenID（关注测试号后获取）
}

# ========== 2. 新增：获取微信access_token ==========
def get_wechat_access_token():
    """获取微信公众号接口调用凭证（有效期2小时，需缓存）"""
    url = f"https://api.weixin.qq.com/cgi-bin/token?grant_type=client_credential&appid={WECHAT_CONFIG['APPID']}&secret={WECHAT_CONFIG['APPSECRET']}"
    try:
        response = requests.get(url, timeout=10)
        result = response.json()
        if "access_token" in result:
            # 缓存token（Demo阶段简单缓存，生产环境用Redis）
            return {
                "token": result["access_token"],
                "expire_time": time.time() + 7200  # 2小时有效期
            }
        else:
            raise Exception(f"获取token失败：{result}")
    except Exception as e:
        raise Exception(f"获取微信token出错：{str(e)}")

# ========== 3. 新增：发送微信客服消息 ==========
def send_wechat_message(openid: str, content: dict):
    """
    发送微信客服消息（文本类型）
    :param openid: 用户OpenID
    :param content: AI分析结果（包含summary、categories等）
    """
    # 1. 获取access_token
    token_info = get_wechat_access_token()
    access_token = token_info["token"]
    
    # 2. 构造消息内容（微信要求的格式）
    message = {
        "touser": openid,
        "msgtype": "text",
        "text": {
            "content": f"""
【AI网页分析结果】
标题：{content['title']}
URL：{content['url']}

📝 摘要：
{content['summary']}

🏷️ 分类标签：
{','.join(content['categories'])}
            """.strip()  # 去掉多余空格
        }
    }
    
    # 3. 调用微信客服消息API
    url = f"https://api.weixin.qq.com/cgi-bin/message/custom/send?access_token={access_token}"
    try:
        response = requests.post(
            url,
            data=json.dumps(message, ensure_ascii=False).encode("utf-8"),
            headers={"Content-Type": "application/json"},
            timeout=10
        )
        result = response.json()
        if result.get("errcode") == 0:
            print("微信消息发送成功！")
            return True
        else:
            raise Exception(f"发送失败：{result}")
    except Exception as e:
        raise Exception(f"微信消息发送出错：{str(e)}")

d:\Courses\AI\envs_py310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:
get_wechat_access_token()

{'token': '101_CUYuIt1ot-_R78f_tep-952HlI0DFxC0gLtX01uSJ6v4zPdHE3Xgp7rVzFQ-TtlpVPNLtAq92bvzi9w_4zVDONIfUikXOTlMYG0XNpWBEMB-itAoeq7ym4CffiIQCPbAEALXK',
 'expire_time': 1772469994.0381262}

In [10]:
import json
import re

def parse_ai_response(ai_response: str) -> dict:
    """
    清理AI返回的JSON字符串并解析
    """
    try:
        cleaned = re.sub(r'```json\s*|\s*```', '', ai_response)
        
        # 解析JSON
        result = json.loads(cleaned.strip())
        return result
        
    except json.JSONDecodeError as e:
        print(f"JSON解析失败: {e}")
        print(f"清理后的内容: {cleaned}")
        # 备用方案：返回默认值
        return {
            "summary": ai_response[:200],
            "categories": ["未分类"]
        }

ai_response = response.choices[0].message.content
parse_ai_response(ai_response)

{'summary': '国家统计局发布的《中华人民共和国2025年国民经济和社会发展统计公报》显示，2025年作为“十四五”规划收官之年，中国经济顶压前行，国内生产总值达140.2万亿元，同比增长5.0%。其中，第三产业增长5.4%，贡献最大。全年经济呈现向新向优发展态势，现代化产业体系建设、改革开放、民生保障等重点领域取得积极进展，社会大局保持稳定，为第二个百年奋斗目标新征程奠定了良好开局。',
 'categories': ['经济', '统计', '政策', '社会发展']}

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import json

# ========== 配置项 ==========
api_key='sk-450c8e6d9d544bea8c251470157548c7'

# ========== 初始化FastAPI应用 ==========
app = FastAPI(title="AI网页分析后端", version="1.0")

# 解决跨域问题（Chrome插件发送请求必须加这个）
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # 生产环境替换成你的插件ID，格式：chrome-extension://插件ID
    allow_credentials=True,
    allow_methods=["GET", "POST", "OPTIONS"],  # 明确允许的方法
    allow_headers=["Content-Type", "Authorization", "Accept"],  # 明确允许的请求头
    expose_headers=["Content-Length"],  # 暴露响应头
)

# 定义请求体结构（和前端发送的参数对应）
class WebContentRequest(BaseModel):
    content: str  # 提取的网页正文
    url: str      # 网页URL
    title: str    # 网页标题

# ========== AI分析核心函数 ==========
def analyze_with_ai(content: str) -> dict:
    """调用AI做摘要和分类"""

    prompt = f"""
    请分析以下网页内容，完成两个任务：
    1. 生成简洁的摘要（300字以内）；
    2. 给出3-5个分类标签（比如：科研、资讯、技术、娱乐、教育等）。
    内容：{content[:5000]}
    输出格式要求：JSON字符串，包含summary和categories两个字段，categories是数组。
    """

    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=1000
        )
        # 解析AI返回结果
        ai_response = response.choices[0].message.content
        return json.loads(ai_response)

    except Exception as e:
        # 备用方案：如果没有AI Key，返回模拟结果
        print(f"AI调用失败：{e}")
        return {
            "summary": f"【模拟摘要】{content[:200]}...",
            "categories": ["科研", "网页内容", "未分类"]
        }

# ========== 核心接口 ==========
@app.post("/api/analyze-web")
async def analyze_web_content(request: WebContentRequest):
    """接收网页内容，返回AI分析结果"""
    try:
        # 1. 验证参数
        if not request.content:
            raise HTTPException(status_code=400, detail="网页内容不能为空")
        
        # 2. 调用AI分析
        ai_result = analyze_with_ai(request.content)
        
        # 3. 返回结果
        return {
            "code": 200,
            "message": "分析成功",
            "summary": ai_result["summary"],
            "categories": ai_result["categories"],
            "url": request.url,
            "title": request.title
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"分析失败：{str(e)}")

# ========== 启动后端 ==========
if __name__ == "__main__":
    import uvicorn
    # 启动服务，监听本地8000端口，允许外部访问
    uvicorn.run("main.py:app", host="0.0.0.0", port=8001, reload=True)

INFO:     Will watch for changes in these directories: ['d:\\Courses\\AI\\agent\\later_read_agent']
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)
INFO:     Started reloader process [6000] using StatReload


INFO:     Stopping reloader process [6000]
